In [1]:
!pip install -q datasets huggingface_hub

In [2]:
from huggingface_hub import login
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")
login(token=HF_TOKEN)

In [3]:
!pip install -q openai datasets pandas tqdm scikit-learn

In [4]:
from google.colab import userdata
import os

os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

In [5]:
from datasets import load_dataset, concatenate_datasets
import pandas as pd

ds_dict = load_dataset("dinushiTJ/nz_research_commons_classification")

if "train" in ds_dict and "test" in ds_dict:
    ds_all = concatenate_datasets([ds_dict["train"], ds_dict["test"]])
else:
    split_name = list(ds_dict.keys())[0]
    ds_all = ds_dict[split_name]

df = ds_all.to_pandas()

print(ds_all)
print(ds_all.features)
print(df.shape)

README.md:   0%|          | 0.00/1.12k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/83.2M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/13271 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3318 [00:00<?, ? examples/s]

Dataset({
    features: ['title', 'authors', 'subjects', 'abstract', 'text', 'record_id_hash', 'prompt', 'token_count', 'classification_label', 'classification_reason', 'year', 'classification_confidence', 'embedding_text', 'embedding_token_count', 'conversations'],
    num_rows: 16589
})
{'title': Value('string'), 'authors': Value('string'), 'subjects': Value('string'), 'abstract': Value('string'), 'text': Value('string'), 'record_id_hash': Value('string'), 'prompt': Value('string'), 'token_count': Value('int32'), 'classification_label': ClassLabel(names=['maori_origin', 'non_maori_origin']), 'classification_reason': Value('string'), 'year': Value('string'), 'classification_confidence': Value('float32'), 'embedding_text': Value('string'), 'embedding_token_count': Value('int64'), 'conversations': List({'role': Value('string'), 'content': Value('string')})}
(16589, 15)


In [6]:
label_col = "classification_label"
label_feature = ds_all.features[label_col]

if hasattr(label_feature, "names"):
    df["classification_label_str"] = df[label_col].apply(
        lambda x: label_feature.int2str(int(x))
    )
else:
    df["classification_label_str"] = df[label_col].astype(str)

mapping_table = (
    df[[label_col, "classification_label_str"]]
    .drop_duplicates()
    .sort_values(label_col)
)

display(mapping_table)
print(df["classification_label_str"].value_counts())

,classification_label,classification_label_str
0,0,maori_origin
1,1,non_maori_origin


classification_label_str
non_maori_origin    14753
maori_origin         1836
Name: count, dtype: int64


In [7]:
N_PER_CLASS = 1000
RANDOM_SEED = 42

maori_sample = df[df["classification_label_str"] == "maori_origin"].sample(
    n=N_PER_CLASS,
    random_state=RANDOM_SEED
)

non_maori_sample = df[df["classification_label_str"] == "non_maori_origin"].sample(
    n=N_PER_CLASS,
    random_state=RANDOM_SEED
)

sample_df = pd.concat([maori_sample, non_maori_sample]).sample(
    frac=1,
    random_state=RANDOM_SEED
).reset_index(drop=True)

sample_df["sample_id"] = sample_df.index

print("Sample shape:", sample_df.shape)
print(sample_df["classification_label_str"].value_counts())

display(
    sample_df.loc[
        [0],
        [
            "sample_id",
            "record_id_hash",
            "classification_label",
            "classification_label_str",
            "title",
            "subjects",
            "abstract",
        ]
    ]
)

Sample shape: (2000, 17)
classification_label_str
non_maori_origin    1000
maori_origin        1000
Name: count, dtype: int64


,sample_id,record_id_hash,classification_label,classification_label_str,title,subjects,abstract
0,0,db3090f92d2c150ee59a1b04e40dc8131005c14d3f402a...,1,non_maori_origin,Child past and future thinking: What children ...,episodic memory,Thinking about memories and possible future ev...


In [8]:
def clean_text(x):
    if x is None:
        return ""
    if pd.isna(x):
        return ""
    return str(x)


def build_prompt(row):
    title = clean_text(row.get("title", ""))
    authors = clean_text(row.get("authors", ""))
    year = clean_text(row.get("year", ""))
    subjects = clean_text(row.get("subjects", ""))
    abstract = clean_text(row.get("abstract", ""))

    return f"""You are a specialist in classifying academic research papers by their relevance to Maori topics.

Classify the academic research paper below into EXACTLY ONE category:
- maori_origin: Substantively connected to Maori people, communities, land, resources, or knowledge
- non_maori_origin: No meaningful Maori connection

QUALIFIES AS maori_origin if it substantively involves ANY of:

1. PEOPLE AND COMMUNITY
   - Maori people, whanau, hapu, or iwi as primary subject or participant group
   - Maori health, education, economic, justice, or social outcomes
   - Maori demographic studies where Maori outcomes are the primary focus

2. CULTURE, IDENTITY AND LANGUAGE
   - Te reo Maori: acquisition, revitalisation, or documentation
   - Maori cultural practices, tikanga, whakapapa, or arts
   - Maori oral traditions, media, or digital representation

3. HISTORY, POLITICS AND LAW
   - Treaty of Waitangi, Maori land rights, Waitangi Tribunal, Treaty settlements
   - Maori sovereignty, tino rangatiratanga, colonisation impacts
   - Legislation or policy primarily affecting Maori communities

4. KNOWLEDGE SYSTEMS AND METHODOLOGY
   - Matauranga Maori as subject or framework
   - Kaupapa Maori or Indigenous research methodology
   - Research co-designed with or led by Maori communities or organisations

5. LAND, ENVIRONMENT AND RESOURCES
   - Research on Maori land, whenua, rohe, or areas of Maori significance
   - Environmental or ecological studies involving Maori waterways, taonga species, mahinga kai sites, or resources under Maori kaitiakitanga
   - Environmental impacts specifically on Maori communities or land

6. ARCHAEOLOGY AND MATERIAL CULTURE
   - Maori sites, pa, urupa, artefacts, taonga, or material culture
   - Genomic or skeletal studies of ancestral Maori populations

7. DATA AND FUNDING CONTEXT
   - Maori-specific datasets, population registries, or data sovereignty
   - Research commissioned or partnered by iwi, hapu, or Maori organisations

DOES NOT qualify, classify as non_maori_origin:
   - Maori as one minor ethnic subgroup in a broader New Zealand population study
   - Maori place names or words used incidentally
   - New Zealand-based research where Maori are not involved, affected, or a focus
   - Passing mention of Maori in introduction or background only

CENTRAL TEST:
   1. Is the research about Maori people, culture, history, rights, or knowledge?
      -> maori_origin
   2. Is the research on Maori land or resources with Maori significance?
      -> maori_origin
   3. Does the research use matauranga Maori or a Maori methodology?
      -> maori_origin
   4. Is the research commissioned or co-designed by a Maori organisation or iwi?
      -> maori_origin
   5. Is the Maori connection only incidental, linguistic, or geographic?
      -> non_maori_origin
   6. Still uncertain?
      -> non_maori_origin

GUIDELINES:
   - Prioritise abstract and subjects.
   - A single strong Maori dimension is sufficient for maori_origin.
   - Ambiguous or peripheral Maori relevance should be classified as non_maori_origin.
   - Confidence should range from 0.0 to 1.0.

OUTPUT:
Return only this JSON, with no other text.

{{
  "label": "maori_origin | non_maori_origin",
  "confidence": 0.85,
  "reason": "Brief explanation of which dimension led to this classification."
}}

RESEARCH PAPER:
Title: {title}
Authors: {authors}
Year: {year}
Subjects: {subjects}
Abstract: {abstract}
"""

In [9]:
from openai import OpenAI
import json
import time

groq_client = OpenAI(
    api_key=os.environ["GROQ_API_KEY"],
    base_url="https://api.groq.com/openai/v1",
)

MODEL_NAME = "llama-3.1-8b-instant"

In [10]:

def classify_with_llama_groq(row, max_retries=5):
    prompt = build_prompt(row)

    for attempt in range(max_retries):
        try:
            response = groq_client.chat.completions.create(
                model=MODEL_NAME,
                messages=[
                    {
                        "role": "system",
                        "content": (
                            "You are a careful academic metadata classifier. "
                            "Return only valid JSON with fields: label, confidence, reason. "
                            "The label must be either maori_origin or non_maori_origin."
                        ),
                    },
                    {
                        "role": "user",
                        "content": prompt,
                    },
                ],
                response_format={"type": "json_object"},
                temperature=0,
                max_tokens=150,
            )

            text = response.choices[0].message.content
            parsed = json.loads(text)

            label = parsed.get("label")
            confidence = parsed.get("confidence")
            reason = parsed.get("reason")

            if label not in ["maori_origin", "non_maori_origin"]:
                raise ValueError(f"Invalid label returned: {label}")

            return {
                "llama_label": label,
                "llama_confidence": float(confidence),
                "llama_reason": str(reason),
                "llama_error": None,
            }

        except Exception as e:
            wait = 20
            print(f"[WARN] attempt {attempt + 1} failed: {e}")
            time.sleep(wait)

    return {
        "llama_label": None,
        "llama_confidence": None,
        "llama_reason": None,
        "llama_error": "failed_after_retries",
    }

In [11]:
for i in range(3):
    result = classify_with_llama_groq(sample_df.iloc[i])

    print("Index:", i)
    print("Original label:", sample_df.iloc[i]["classification_label_str"])
    print("Llama/Groq result:", result)
    print()

Index: 0
Original label: non_maori_origin
Llama/Groq result: {'llama_label': 'non_maori_origin', 'llama_confidence': 0.95, 'llama_reason': 'The research paper focuses on child development and cognitive processes, with no apparent connection to Maori people, culture, history, rights, or knowledge.', 'llama_error': None}

Index: 1
Original label: maori_origin
Llama/Groq result: {'llama_label': 'maori_origin', 'llama_confidence': 0.95, 'llama_reason': 'The research focuses on te reo Māori teachers and draws on the epistemology of Māori language and Māori language learning, indicating a strong connection to Maori culture and knowledge systems.', 'llama_error': None}

Index: 2
Original label: non_maori_origin
Llama/Groq result: {'llama_label': 'non_maori_origin', 'llama_confidence': 1.0, 'llama_reason': 'The research paper is about chemistry and does not involve Maori people, culture, history, rights, or knowledge.', 'llama_error': None}



In [12]:
from datasets import load_dataset, Dataset
from pathlib import Path
import pandas as pd
import time
from tqdm.auto import tqdm

# Save checkpoint to Google Drive
CHECKPOINT_DIR = Path("/content/drive/MyDrive/maori_data_finder_llama_groq_checkpoints")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

checkpoint_path = CHECKPOINT_DIR / "llama_groq_label_comparison_checkpoint.csv"

HF_REPO_ID = "dinushiTJ/nz-research-commons-llama-groq-label-comparison"
PUSH_EVERY = 10

KEEP_COLS = [
    "record_id_hash",
    "sample_id",

    "classification_label",
    "classification_label_str",
    "classification_confidence",
    "classification_reason",

    "title",
    "authors",
    "year",
    "subjects",
    "abstract",

    "llama_label",
    "llama_confidence",
    "llama_reason",
    "llama_error",
    "exact_match",

    "comparison_provider",
    "comparison_model",
    "comparison_type",
    "prompt_version",
    "sample_type",
    "n_per_class",
    "random_seed",
]

def clean_for_hf(df):
    existing_cols = [c for c in KEEP_COLS if c in df.columns]
    return df[existing_cols].copy()


# 1. Try Google Drive checkpoint first
if checkpoint_path.exists():
    comparison_results = pd.read_csv(checkpoint_path)
    print(f"Loaded checkpoint from Google Drive: {len(comparison_results)} rows")

# 2. If Drive checkpoint does not exist, try Hugging Face
else:
    try:
        hf_ds = load_dataset(HF_REPO_ID)

        if "train" in hf_ds:
            comparison_results = hf_ds["train"].to_pandas()
        else:
            split_name = list(hf_ds.keys())[0]
            comparison_results = hf_ds[split_name].to_pandas()

        print(f"Loaded checkpoint from Hugging Face: {len(comparison_results)} rows")

        # Save Hugging Face checkpoint back to Drive
        comparison_results.to_csv(checkpoint_path, index=False)
        print("Saved Hugging Face checkpoint to Google Drive.")

    except Exception as e:
        print("No Drive checkpoint or Hugging Face checkpoint found.")
        print("Starting fresh.")
        comparison_results = pd.DataFrame()


# Build completed ID set
if len(comparison_results) > 0 and "record_id_hash" in comparison_results.columns:
    completed_ids = set(comparison_results["record_id_hash"].astype(str))
    rows = comparison_results.to_dict("records")
else:
    completed_ids = set()
    rows = []

print("Already completed rows:", len(rows))
print("Current sample size:", len(sample_df))
print(
    "Already completed in this sample:",
    sample_df["record_id_hash"].astype(str).isin(completed_ids).sum()
)
print(
    "Remaining to run:",
    (~sample_df["record_id_hash"].astype(str).isin(completed_ids)).sum()
)

README.md:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/2.54M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Loaded checkpoint from Hugging Face: 2000 rows
Saved Hugging Face checkpoint to Google Drive.
Already completed rows: 2000
Current sample size: 2000
Already completed in this sample: 2000
Remaining to run: 0


In [13]:
for _, row in tqdm(sample_df.iterrows(), total=len(sample_df)):
    record_id = str(row["record_id_hash"])

    if record_id in completed_ids:
        continue

    result = classify_with_llama_groq(row)

    output_row = row.to_dict()
    output_row.update(result)

    output_row["exact_match"] = (
        output_row["classification_label_str"] == output_row["llama_label"]
    )

    output_row["comparison_provider"] = "Groq"
    output_row["comparison_model"] = MODEL_NAME
    output_row["comparison_type"] = "cross-provider_label_consistency_check"
    output_row["prompt_version"] = "same_as_gemini_reference_prompt"
    output_row["sample_type"] = "balanced_by_class"
    output_row["n_per_class"] = N_PER_CLASS
    output_row["random_seed"] = RANDOM_SEED

    rows.append(output_row)
    completed_ids.add(record_id)

    # Save full checkpoint to Drive after every record
    comparison_results = pd.DataFrame(rows)
    comparison_results.to_csv(checkpoint_path, index=False)

    # Push clean version to Hugging Face every PUSH_EVERY records
    if len(rows) % PUSH_EVERY == 0:
        clean_comparison_results = clean_for_hf(comparison_results)

        comparison_hf = Dataset.from_pandas(
            clean_comparison_results,
            preserve_index=False
        )

        comparison_hf.push_to_hub(HF_REPO_ID)

        print(f"Saved checkpoint and pushed {len(rows)} rows to Hugging Face.")

    time.sleep(25)


# Final save and push
comparison_results = pd.DataFrame(rows)
comparison_results.to_csv(checkpoint_path, index=False)

clean_comparison_results = clean_for_hf(comparison_results)

comparison_hf = Dataset.from_pandas(
    clean_comparison_results,
    preserve_index=False
)

comparison_hf.push_to_hub(HF_REPO_ID)

print("Comparison complete.")
print("Local checkpoint:", checkpoint_path)
print("Hugging Face repo:", HF_REPO_ID)
print("Total completed rows:", len(comparison_results))

  0%|          | 0/2000 [00:00<?, ?it/s]

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/2 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

                              :  88%|########8 | 2.23MB / 2.54MB            

No files have been modified since last commit. Skipping to prevent empty commit.


Comparison complete.
Local checkpoint: /content/drive/MyDrive/maori_data_finder_llama_groq_checkpoints/llama_groq_label_comparison_checkpoint.csv
Hugging Face repo: dinushiTJ/nz-research-commons-llama-groq-label-comparison
Total completed rows: 2000


In [14]:
comparison_results = pd.read_csv(checkpoint_path)

print("Completed rows:", len(comparison_results))
print("Successful labels:", comparison_results["llama_label"].notna().sum())
print("Failed labels:", comparison_results["llama_label"].isna().sum())

Completed rows: 2000
Successful labels: 1972
Failed labels: 28


In [15]:
clean_comparison_results = clean_for_hf(comparison_results)

comparison_hf = Dataset.from_pandas(
    clean_comparison_results,
    preserve_index=False
)

comparison_hf.push_to_hub(HF_REPO_ID)

print("Current checkpoint pushed to Hugging Face.")

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/2 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

                              : 100%|##########| 2.54MB / 2.54MB            

No files have been modified since last commit. Skipping to prevent empty commit.


Current checkpoint pushed to Hugging Face.


In [16]:
keep_cols = [
    "record_id_hash",
    "sample_id",
    "classification_label",
    "classification_label_str",
    "classification_confidence",
    "classification_reason",
    "title",
    "authors",
    "year",
    "subjects",
    "abstract",
    "llama_label",
    "llama_confidence",
    "llama_reason",
    "llama_error",
    "exact_match",
    "comparison_provider",
    "comparison_model",
    "comparison_type",
    "prompt_version",
    "sample_type",
    "n_per_class",
    "random_seed",
]

clean_comparison_results = comparison_results[keep_cols].copy()

clean_comparison_results.head()

,record_id_hash,sample_id,classification_label,classification_label_str,classification_confidence,classification_reason,title,authors,year,subjects,...,llama_reason,llama_error,exact_match,comparison_provider,comparison_model,comparison_type,prompt_version,sample_type,n_per_class,random_seed
0,a552a090b1d3dc916b563e15751e1d221311b5f3dc0db7...,0,0,maori_origin,1.00,The research is written in Te Reo Māori and fo...,"Te Mata Ono. E tipu atu te kūmara, e ohu mai t...","Pēwhairangi, Te Aorere Tūwhakairiora",2019,Māori television,...,"The research paper is about Māori television, ...",NaN,True,Groq,llama-3.1-8b-instant,cross-provider_label_consistency_check,same_as_gemini_reference_prompt,balanced_by_class,10,42
1,ab836bcbe8e20c1fdb98c8cc5de6ea04e40811e15ddabc...,1,1,non_maori_origin,0.95,The research evaluates a general New Zealand l...,The measurement of training opportunities cour...,"Stolte, Ottilie Emma Elisabeth",2004,NaN,...,The research paper focuses on labour market po...,NaN,True,Groq,llama-3.1-8b-instant,cross-provider_label_consistency_check,same_as_gemini_reference_prompt,balanced_by_class,10,42
2,b39870c157608b0181537722a85c48b49d5e76555ba807...,2,1,non_maori_origin,1.00,The paper is an introduction to a special issu...,Introduction: Special Issue of Selected Papers...,"Holmes, Geoffrey",2017,computer science,...,The research paper is about a computer science...,NaN,True,Groq,llama-3.1-8b-instant,cross-provider_label_consistency_check,same_as_gemini_reference_prompt,balanced_by_class,10,42
3,23950016fc39c593bc7fc201b7409a86908b4e7f2a4166...,3,0,maori_origin,1.00,The research substantively focuses on Māori or...,Revitalizing Te Ika-a-Maui: Māori migration an...,"Mahuika, Nepia",2009,Māori,...,The research paper is primarily about Māori my...,NaN,True,Groq,llama-3.1-8b-instant,cross-provider_label_consistency_check,same_as_gemini_reference_prompt,balanced_by_class,10,42
4,2499be3959af8bd5fc3d0d5a66c02fd87705bebe40380a...,4,0,maori_origin,1.00,The research focuses specifically on the Pōwhi...,Pōwhiri process in mental health research,"McClintock, Kahu",2010,indigenous mental health research,...,"The research focuses on the Pōwhiri process, a...",NaN,True,Groq,llama-3.1-8b-instant,cross-provider_label_consistency_check,same_as_gemini_reference_prompt,balanced_by_class,10,42


In [17]:
from datasets import Dataset

comparison_hf = Dataset.from_pandas(
    clean_comparison_results,
    preserve_index=False
)

comparison_hf.push_to_hub(
    "dinushiTJ/nz-research-commons-llama-groq-label-comparison"
)

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/2 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

                              : 100%|##########| 2.54MB / 2.54MB            

No files have been modified since last commit. Skipping to prevent empty commit.


CommitInfo(commit_url='https://huggingface.co/datasets/dinushiTJ/nz-research-commons-llama-groq-label-comparison/commit/432eda28c3c0f24944992e61ba18f0f793761042', commit_message='Upload dataset', commit_description='', oid='432eda28c3c0f24944992e61ba18f0f793761042', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/dinushiTJ/nz-research-commons-llama-groq-label-comparison', endpoint='https://huggingface.co', repo_type='dataset', repo_id='dinushiTJ/nz-research-commons-llama-groq-label-comparison'), pr_revision=None, pr_num=None)

In [18]:
mismatch_table = clean_comparison_results[
    (clean_comparison_results["llama_label"].notna()) &
    (clean_comparison_results["classification_label_str"] != clean_comparison_results["llama_label"])
].copy()

mismatch_hf = Dataset.from_pandas(
    mismatch_table,
    preserve_index=False
)

mismatch_hf.push_to_hub(
    "dinushiTJ/nz-research-commons-llama-groq-label-mismatches"
)

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

                              : 100%|##########|  166kB /  166kB            

README.md:   0%|          | 0.00/1.20k [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/datasets/dinushiTJ/nz-research-commons-llama-groq-label-mismatches/commit/a128657204dd605c5c8143da2a20f4c344f5a51a', commit_message='Upload dataset', commit_description='', oid='a128657204dd605c5c8143da2a20f4c344f5a51a', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/dinushiTJ/nz-research-commons-llama-groq-label-mismatches', endpoint='https://huggingface.co', repo_type='dataset', repo_id='dinushiTJ/nz-research-commons-llama-groq-label-mismatches'), pr_revision=None, pr_num=None)

In [19]:
exact_verify_table = comparison_results[
    [
        "sample_id",
        "record_id_hash",
        "classification_label",
        "classification_label_str",
        "llama_label",
        "llama_confidence",
        "exact_match",
        "llama_reason",
        "title",
    ]
].copy()

exact_verify_table.columns = [
    "Sample ID",
    "Record ID",
    "Original numeric label",
    "Original text label",
    "Llama/Groq label",
    "Llama/Groq confidence",
    "Exact match",
    "Llama/Groq reason",
    "Title",
]

display(exact_verify_table)

,Sample ID,Record ID,Original numeric label,Original text label,Llama/Groq label,Llama/Groq confidence,Exact match,Llama/Groq reason,Title
0,0,a552a090b1d3dc916b563e15751e1d221311b5f3dc0db7...,0,maori_origin,maori_origin,0.95,True,"The research paper is about Māori television, ...","Te Mata Ono. E tipu atu te kūmara, e ohu mai t..."
1,1,ab836bcbe8e20c1fdb98c8cc5de6ea04e40811e15ddabc...,1,non_maori_origin,non_maori_origin,0.95,True,The research paper focuses on labour market po...,The measurement of training opportunities cour...
2,2,b39870c157608b0181537722a85c48b49d5e76555ba807...,1,non_maori_origin,non_maori_origin,1.00,True,The research paper is about a computer science...,Introduction: Special Issue of Selected Papers...
3,3,23950016fc39c593bc7fc201b7409a86908b4e7f2a4166...,0,maori_origin,maori_origin,0.95,True,The research paper is primarily about Māori my...,Revitalizing Te Ika-a-Maui: Māori migration an...
4,4,2499be3959af8bd5fc3d0d5a66c02fd87705bebe40380a...,0,maori_origin,maori_origin,0.95,True,"The research focuses on the Pōwhiri process, a...",Pōwhiri process in mental health research
...,...,...,...,...,...,...,...,...,...
1995,1954,23bf83871506ed1d492a18ee2cefdddc338138a09f39da...,0,maori_origin,maori_origin,0.95,True,The research paper is about Maori philosophy a...,The responsibility towards indigenous holism: ...
1996,1961,6b014e60e23759a4605971b17b4be79cbd48e3d692f8a1...,1,non_maori_origin,NaN,NaN,False,NaN,Biosensing-by-Learning Direct Targeting Strate...
1997,1973,0c5e89068dbe7658b09ac7eb05d8182bade4f4e0849433...,0,maori_origin,maori_origin,0.95,True,The research paper is about Māori data soverei...,"Māori, census 2018 and data sovereignty"
1998,1975,37a01eaddc15966456cf0a371fb0cc82fc7af5aef7fa7e...,1,non_maori_origin,NaN,NaN,False,NaN,The relevance of calf muscle metrics and plyom...


In [20]:
total_records = len(comparison_results)
successful_records = comparison_results["llama_label"].notna().sum()
failed_records = comparison_results["llama_label"].isna().sum()

matched_records = comparison_results["exact_match"].sum()
mismatched_records = successful_records - matched_records

summary_table = pd.DataFrame([
    {
        "Metric": "Total sampled records",
        "Count": total_records,
        "Percentage": 100.00,
    },
    {
        "Metric": "Successfully classified by Llama/Groq",
        "Count": successful_records,
        "Percentage": successful_records / total_records * 100,
    },
    {
        "Metric": "Failed / missing Llama/Groq label",
        "Count": failed_records,
        "Percentage": failed_records / total_records * 100,
    },
    {
        "Metric": "Exact matches",
        "Count": matched_records,
        "Percentage": matched_records / successful_records * 100 if successful_records > 0 else 0,
    },
    {
        "Metric": "Mismatches",
        "Count": mismatched_records,
        "Percentage": mismatched_records / successful_records * 100 if successful_records > 0 else 0,
    },
])

summary_table["Percentage"] = summary_table["Percentage"].round(2)

display(summary_table)

,Metric,Count,Percentage
0,Total sampled records,2000,100.00
1,Successfully classified by Llama/Groq,1972,98.60
2,Failed / missing Llama/Groq label,28,1.40
3,Exact matches,1865,94.57
4,Mismatches,107,5.43


In [21]:
label_comparison_counts = pd.crosstab(
    comparison_results["classification_label_str"],
    comparison_results["llama_label"],
    rownames=["Original text label"],
    colnames=["Llama/Groq label"],
    margins=True,
)

display(label_comparison_counts)

Llama/Groq label,maori_origin,non_maori_origin,All
Original text label,,,
maori_origin,900,88,988
non_maori_origin,19,965,984
All,919,1053,1972


# failed rows

In [22]:
failed_rows = comparison_results[
    comparison_results["llama_label"].isna()
].copy()

print("Failed rows to rerun:", len(failed_rows))
display(
    failed_rows[
        [
            "record_id_hash",
            "classification_label_str",
            "llama_label",
            "llama_error",
            "title"
        ]
    ].head()
)

Failed rows to rerun: 28


,record_id_hash,classification_label_str,llama_label,llama_error,title
1952,9476bcd81cacc7f1f021e6f5714d5f271b4e258ecf1cff...,non_maori_origin,NaN,failed_after_retries,Novel New Zealand marine bioactive natural pro...
1954,f30c81eea3942e91a356adb768d7bd37a7e9308d4dd592...,non_maori_origin,NaN,failed_after_retries,The effects of mask-wearing and teleneuropsych...
1955,d34ac86af0d231b3774ff8da79c1b4a3d60c04052ca61c...,non_maori_origin,NaN,failed_after_retries,Evaluation of effectiveness of the working at ...
1957,1decb0a5eac90097baa0963b00c479c226b2fa90c3f803...,maori_origin,NaN,failed_after_retries,Contextualising Māori writing: a study of pros...
1959,29cc8e4be78d2c034a1bd94578dd29c99c06ab9ee0ec53...,maori_origin,NaN,failed_after_retries,An evaluation of Kaupapa Māori in Psychology a...


In [23]:
rerun_rows = []

for _, row in tqdm(failed_rows.iterrows(), total=len(failed_rows)):
    result = classify_with_llama_groq(row)

    updated_row = row.to_dict()
    updated_row.update(result)

    updated_row["exact_match"] = (
        updated_row["classification_label_str"] == updated_row["llama_label"]
    )

    rerun_rows.append(updated_row)

    time.sleep(15)

  0%|          | 0/28 [00:00<?, ?it/s]

In [24]:
successful_old = comparison_results[
    comparison_results["llama_label"].notna()
].copy()

rerun_results = pd.DataFrame(rerun_rows)

comparison_results_updated = pd.concat(
    [successful_old, rerun_results],
    ignore_index=True
)

print("Old successful rows:", len(successful_old))
print("Rerun rows:", len(rerun_results))
print("Updated total rows:", len(comparison_results_updated))
print("Still failed:", comparison_results_updated["llama_label"].isna().sum())

Old successful rows: 1972
Rerun rows: 28
Updated total rows: 2000
Still failed: 0


In [25]:
comparison_results_updated.to_csv(checkpoint_path, index=False)

clean_comparison_results = clean_for_hf(comparison_results_updated)

comparison_hf = Dataset.from_pandas(
    clean_comparison_results,
    preserve_index=False
)

comparison_hf.push_to_hub(HF_REPO_ID)

print("Updated comparison pushed to Hugging Face.")

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/2 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

                              :  80%|########  | 2.04MB / 2.54MB            

Updated comparison pushed to Hugging Face.


In [26]:
mismatch_table = comparison_results[
    (comparison_results["llama_label"].notna()) &
    (comparison_results["classification_label_str"] != comparison_results["llama_label"])
].copy()

display(
    mismatch_table[
        [
            "sample_id",
            "record_id_hash",
            "classification_label",
            "classification_label_str",
            "llama_label",
            "llama_confidence",
            "llama_reason",
            "title",
            "subjects",
            "abstract",
        ]
    ]
)

,sample_id,record_id_hash,classification_label,classification_label_str,llama_label,llama_confidence,llama_reason,title,subjects,abstract
19,19,ee3287229f56cee562bd0096fca96d78fdd02db896bdd4...,0,maori_origin,non_maori_origin,0.95,The research focuses on the effectiveness of m...,Nga Wahi Ipurangi - Digital Places,mLearning,"The Waipa district, unbeknown to many New Zeal..."
20,0,a53ead8f037d7813df267bec29e280cefd14a4c4d37763...,0,maori_origin,non_maori_origin,0.95,The research paper focuses on the physical cha...,The relationship between on-water performance ...,NaN,Objectives: The aim of the present study was t...
23,4,a2c1f1bee658d9a9c9d8591d0ec2f1b8e066a88e1cd238...,0,maori_origin,non_maori_origin,0.95,The research paper is about fish abundance in ...,Comparative fish abundance in the shallow Waik...,NaN,The Waikato-Tainui College for Research and De...
35,31,e517c27ee185d6c58131c4fd50450549f9d41066587512...,0,maori_origin,non_maori_origin,0.95,The research focuses on cross-cultural communi...,Working across cultures in indigenous science ...,cross-cultural,The research in this thesis considers the abil...
99,55,3f71159d55d98b610cb942608dd92b433421ee5ce553de...,0,maori_origin,non_maori_origin,0.95,The research paper focuses on underwater sound...,Spatial and temporal characteristics of the un...,Underwater Soundscapes,Marine ecosystems around the world have been i...
...,...,...,...,...,...,...,...,...,...,...
1943,1919,6aafba66d5a68985b2e70320aedcecf31785a3e0308f95...,0,maori_origin,non_maori_origin,0.95,"The research focuses on Pākehā identity, authe...",Pākehā Roots: Is claiming a Pākehā identity ba...,Authenticity,Development of a Pākehā identity has been an o...
1947,1951,6e419d33aaab9f96b0aa3271a17cd090d128f01448cfd4...,0,maori_origin,non_maori_origin,0.95,The research focuses on the comparative cognit...,The general cognitive ability of Maori and Eur...,NaN,Research was undertaken into the performances ...
1973,462,f201c67ab19e9b24052034ecc69efca8e44dc6cffe7d85...,0,maori_origin,non_maori_origin,0.80,The research focuses on patient experiences an...,Exploring patient experiences and power dynami...,Nursing,"Background: Alongside health outcomes, patient..."
1982,1845,43efb1952c3ff9bbaa708767b91c61871b19413dcbeb42...,0,maori_origin,non_maori_origin,0.95,The research paper is about sustainability val...,Spill-over of sustainability values and practi...,Maori,The following presentation relates to my propo...


end

In [28]:
comparison_results = pd.read_csv(checkpoint_path)

print("Completed rows:", len(comparison_results))
print("Successful labels:", comparison_results["llama_label"].notna().sum())
print("Failed labels:", comparison_results["llama_label"].isna().sum())

Completed rows: 2000
Successful labels: 2000
Failed labels: 0


In [30]:
import pandas as pd
from datasets import load_dataset

HF_REPO_ID = "dinushiTJ/nz-research-commons-llama-groq-label-comparison"

# Load Hugging Face dataset
ds = load_dataset(HF_REPO_ID)

README.md:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/2.54M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [32]:
print(ds)

DatasetDict({
    train: Dataset({
        features: ['record_id_hash', 'sample_id', 'classification_label', 'classification_label_str', 'classification_confidence', 'classification_reason', 'title', 'authors', 'year', 'subjects', 'abstract', 'llama_label', 'llama_confidence', 'llama_reason', 'llama_error', 'exact_match', 'comparison_provider', 'comparison_model', 'comparison_type', 'prompt_version', 'sample_type', 'n_per_class', 'random_seed'],
        num_rows: 2000
    })
})


In [34]:
comparison_results = ds["train"].to_pandas()

comparison_results.head()

,record_id_hash,sample_id,classification_label,classification_label_str,classification_confidence,classification_reason,title,authors,year,subjects,...,llama_reason,llama_error,exact_match,comparison_provider,comparison_model,comparison_type,prompt_version,sample_type,n_per_class,random_seed
0,a552a090b1d3dc916b563e15751e1d221311b5f3dc0db7...,0,0,maori_origin,1.00,The research is written in Te Reo Māori and fo...,"Te Mata Ono. E tipu atu te kūmara, e ohu mai t...","Pēwhairangi, Te Aorere Tūwhakairiora",2019,Māori television,...,"The research paper is about Māori television, ...",None,True,Groq,llama-3.1-8b-instant,cross-provider_label_consistency_check,same_as_gemini_reference_prompt,balanced_by_class,10,42
1,ab836bcbe8e20c1fdb98c8cc5de6ea04e40811e15ddabc...,1,1,non_maori_origin,0.95,The research evaluates a general New Zealand l...,The measurement of training opportunities cour...,"Stolte, Ottilie Emma Elisabeth",2004,None,...,The research paper focuses on labour market po...,None,True,Groq,llama-3.1-8b-instant,cross-provider_label_consistency_check,same_as_gemini_reference_prompt,balanced_by_class,10,42
2,b39870c157608b0181537722a85c48b49d5e76555ba807...,2,1,non_maori_origin,1.00,The paper is an introduction to a special issu...,Introduction: Special Issue of Selected Papers...,"Holmes, Geoffrey",2017,computer science,...,The research paper is about a computer science...,None,True,Groq,llama-3.1-8b-instant,cross-provider_label_consistency_check,same_as_gemini_reference_prompt,balanced_by_class,10,42
3,23950016fc39c593bc7fc201b7409a86908b4e7f2a4166...,3,0,maori_origin,1.00,The research substantively focuses on Māori or...,Revitalizing Te Ika-a-Maui: Māori migration an...,"Mahuika, Nepia",2009,Māori,...,The research paper is primarily about Māori my...,None,True,Groq,llama-3.1-8b-instant,cross-provider_label_consistency_check,same_as_gemini_reference_prompt,balanced_by_class,10,42
4,2499be3959af8bd5fc3d0d5a66c02fd87705bebe40380a...,4,0,maori_origin,1.00,The research focuses specifically on the Pōwhi...,Pōwhiri process in mental health research,"McClintock, Kahu",2010,indigenous mental health research,...,"The research focuses on the Pōwhiri process, a...",None,True,Groq,llama-3.1-8b-instant,cross-provider_label_consistency_check,same_as_gemini_reference_prompt,balanced_by_class,10,42


In [35]:
total_records = len(comparison_results)

successful_records = comparison_results["llama_label"].notna().sum()
failed_records = comparison_results["llama_label"].isna().sum()

matched_records = comparison_results["exact_match"].sum()
mismatched_records = successful_records - matched_records

summary_table = pd.DataFrame([
    {
        "Metric": "Total sampled records",
        "Count": total_records,
        "Percentage": 100.00,
    },
    {
        "Metric": "Successfully classified by Llama/Groq",
        "Count": successful_records,
        "Percentage": successful_records / total_records * 100,
    },
    {
        "Metric": "Failed / missing Llama/Groq label",
        "Count": failed_records,
        "Percentage": failed_records / total_records * 100,
    },
    {
        "Metric": "Exact matches",
        "Count": matched_records,
        "Percentage": matched_records / successful_records * 100 if successful_records > 0 else 0,
    },
    {
        "Metric": "Mismatches",
        "Count": mismatched_records,
        "Percentage": mismatched_records / successful_records * 100 if successful_records > 0 else 0,
    },
])

summary_table["Percentage"] = summary_table["Percentage"].round(2)

display(summary_table)

,Metric,Count,Percentage
0,Total sampled records,2000,100.00
1,Successfully classified by Llama/Groq,2000,100.00
2,Failed / missing Llama/Groq label,0,0.00
3,Exact matches,1891,94.55
4,Mismatches,109,5.45
